In [3]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

from pathlib import Path

import torch
import torch.nn as nn

from torchvision import datasets
from torchvision import transforms
from torchvision import models

from torch.utils.data import DataLoader

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [4]:
# ============================================================
# DEVICE AND PATHS
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using Device :", device)

PROJECT_ROOT = Path.cwd().parent

TEST_DIR = PROJECT_ROOT / "data" / "test"

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "railway_resnet18_transfer.pth"
)

print("Test Path :", TEST_DIR)
print("Model Path :", MODEL_PATH)

Using Device : cuda
Test Path : d:\Railway_AI_Inspector\data\test
Model Path : d:\Railway_AI_Inspector\models\railway_resnet18_transfer.pth


In [5]:
# ============================================================
# TEST TRANSFORMATIONS
# ============================================================

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Test Transformations Created Successfully")

Test Transformations Created Successfully


In [6]:
# ============================================================
# LOAD TEST DATASET
# ============================================================

test_dataset = datasets.ImageFolder(
    root=TEST_DIR,
    transform=test_transform
)

print("Test Samples :", len(test_dataset))

print("\nClasses:")
print(test_dataset.classes)

print("\nClass Mapping:")
print(test_dataset.class_to_idx)

Test Samples : 15

Classes:
['broken_rail', 'crack', 'misalignment', 'normal', 'surface_wear']

Class Mapping:
{'broken_rail': 0, 'crack': 1, 'misalignment': 2, 'normal': 3, 'surface_wear': 4}


In [7]:
# ============================================================
# TEST DATALOADER
# ============================================================

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

print("Test DataLoader Created Successfully")

Test DataLoader Created Successfully


In [8]:
# ============================================================
# REBUILD RESNET18
# ============================================================

resnet18 = models.resnet18(
    weights=None
)

num_features = resnet18.fc.in_features

resnet18.fc = nn.Linear(
    num_features,
    5
)

resnet18 = resnet18.to(device)

print("ResNet18 Architecture Recreated")

ResNet18 Architecture Recreated


In [9]:
# ============================================================
# LOAD SAVED MODEL
# ============================================================

resnet18.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device
    )
)

resnet18.eval()

print("Transfer Learning Model Loaded Successfully")

Transfer Learning Model Loaded Successfully


In [10]:
# ============================================================
# TEST EVALUATION
# ============================================================

all_labels = []
all_predictions = []

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = resnet18(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

test_accuracy = 100 * correct / total

print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 53.33%


In [11]:
# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[1 1 0 1 0]
 [0 0 0 1 2]
 [0 0 2 1 0]
 [1 0 0 2 0]
 [0 0 0 0 3]]


In [ ]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=test_dataset.classes
    )
)

              precision    recall  f1-score   support

 broken_rail       0.50      0.33      0.40         3
       crack       0.00      0.00      0.00         3
misalignment       1.00      0.67      0.80         3
      normal       0.40      0.67      0.50         3
surface_wear       0.60      1.00      0.75         3

    accuracy                           0.53        15
   macro avg       0.50      0.53      0.49        15
weighted avg       0.50      0.53      0.49        15

